# Custom wake-word trainer — bulletproof Colab notebook (2026 edition)

Train your own openWakeWord model in ~75-90 minutes on Colab Pro. **Run all → walk away → download `.onnx` + `.tflite`** at the end.

## Why this notebook exists

openWakeWord's official Colab notebook bit-rotted hard against Python 3.12 (no piper-phonemize wheels), against torchaudio 2.x (`set_audio_backend` removed), against newer Colab images (HF Hub timeouts, namespace-package edge cases), and against its own internal config schema (4 keys silently moved to required-or-`KeyError` over the past year).

This notebook is patched against all of that. Every cell is **self-healing** — checks its own outputs, re-creates what's missing. There's a hard pre-flight gate (cell 3) that catches every dep/file issue before any of the slow downloads start. Each cell also **guards against out-of-order execution** — running a cell before its predecessor will fail immediately with a clear message telling you which cell to run first.

**Multilingual TTS** — set `LANGUAGE` in cell 0 to any Piper locale (`en_US`, `es_ES`, `de_DE`, `fr_FR`, ...). English uses the 904-speaker libritts `.pt` model. All other languages auto-discover and download Piper `.onnx` voices from the [rhasspy/piper-voices](https://huggingface.co/rhasspy/piper-voices) catalog (36 languages).

It also **replaces openwakeword's `auto_train` with a faithful hand-rolled PyTorch loop** that mirrors auto_train's curriculum exactly: 3-stage learning rate (1e-4 → 1e-5 → 1e-6), negative-weight ramp 1→1500, hard-negative mining, FP-per-hour validation against an ACAV100M continuous-audio slice, 90/90/10 percentile checkpoint ensemble averaging. The upstream `auto_train` path keeps surfacing bugs against `mmap_batch_generator` shape handling on current openwakeword `master`; the hand-rolled trainer in cell 7 is small enough to read in one sitting and debug if anything ever changes.

## Run order

1. **Runtime → Change runtime type → L4 GPU + High RAM** (Colab Pro, $10/mo). Free T4 also works but is ~2× slower.
2. **Cell 0 — edit `TARGET_PHRASE`, `MODEL_NAME`, and `LANGUAGE`** for your wake word. (Default = `mr graves` in `en_US`.)
3. **Runtime → Run all** (or run each cell in order — each cell checks that the previous one ran)
4. Walk away ~75-90 min. Last cell downloads `<MODEL_NAME>.onnx` and `<MODEL_NAME>.tflite`.

## Validated example (Mr Graves, 2026-05-09)

13 real-world utterances on a Pixel — every utterance fired:

```
WAKE — score=0.99664426
WAKE — score=0.59243464
WAKE — score=0.65439160
WAKE — score=0.77693284
WAKE — score=0.85433600
WAKE — score=0.64701974
WAKE — score=0.80747485
WAKE — score=0.95243360
WAKE — score=0.99938180
WAKE — score=0.90104705
WAKE — score=0.99639570
WAKE — score=0.98879445
WAKE — score=0.51673140
```

13/13 fires, scores 0.52-0.999, **zero phantom fires** during 30 min of normal phone use (TV, music, conversation, kitchen noise). Matches openwakeword's pretrained Hey Jarvis baseline (0.984/0.989) on clear utterances.

For deeper context on what auto_train actually does and why every part of the curriculum matters, the comments in cell 7 walk through each piece.

## 0. Configure your wake word

Edit the three variables below, then **Runtime → Run all** (or run each cell in order).

In [ ]:
import re
TARGET_PHRASE = ['mr graves', 'mister graves']
MODEL_NAME    = 'mr_graves'
LANGUAGE      = 'en_US'  # Any Piper locale: en_US, es_ES, es_MX, de_DE, fr_FR, ...
                          # Full list: https://rhasspy.github.io/piper-samples/

assert re.match(r'^[a-z]{2}_[A-Z]{2}$', LANGUAGE), \
    f'LANGUAGE must be a Piper locale like en_US or es_ES, got: "{LANGUAGE}"'

_STEP_0_OK = True

## 1. Install dependencies (~2 min)

apt + pip in correct order. `piper-phonemize-cross` first, `piper-tts --no-deps` last.

In [ ]:
assert '_STEP_0_OK' in dir(), '⚠️ Run cell 0 (Config) first!'

# Native deps
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# Python deps — installed in this exact order:
#   (a) piper-phonemize-cross FIRST (Py3.12-compatible fork; same module name as piper-phonemize)
#   (b) Everything openwakeword.train / data.py / utils.py touch (ALL transitives, not just direct)
#   (c) piper-tts LAST via --no-deps (so nothing later can downgrade/clobber it)
#   (d) piper-sample-generator (v3+, supports .onnx Piper voices for multilingual TTS)
!pip install -q piper-phonemize-cross
!pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub
!pip install -q --no-deps piper-tts
!pip install -q piper-sample-generator

# Verify imports — every module openwakeword's train.py, data.py and utils.py
# touch. If anything's missing this surfaces NOW, before any 75-min run.
import sys
print('Python:', sys.version)
import torch, torchinfo, torchmetrics, scipy, numpy
print(f'  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests
from piper_phonemize import phonemize_espeak
from piper import PiperVoice, SynthesisConfig
import piper_sample_generator
print('  All openwakeword deps (incl. piper-tts, piper-sample-generator) import cleanly.')

_STEP_1_OK = True

## 2. Clone openwakeword + download voices + apply patches (~2 min)

Clones openwakeword, downloads TTS voice models for your `LANGUAGE`, and applies runtime patches.

In [ ]:
assert '_STEP_1_OK' in dir(), '⚠️ Run cell 1 (Install) first!'

# ── Clone openwakeword ─────────────────────────────────────────────────────────

import os, sys
os.chdir('/content')

# ── piper-sample-generator directory (for shim + model storage) ────
PSG_DIR = '/content/piper-sample-generator'
os.makedirs(f'{PSG_DIR}/models', exist_ok=True)
print(f'  ✓ piper-sample-generator dir: {PSG_DIR}')

# ── openwakeword (master) ─────────────────────────────────────────────────────
OWW_DIR = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'
if not os.path.exists(OWW_TRAIN):
    print('  openwakeword missing or gutted — re-cloning')
    !rm -rf {OWW_DIR}
    !git clone -q https://github.com/dscripka/openwakeword {OWW_DIR}
    !pip install -q -e {OWW_DIR}
assert os.path.exists(OWW_TRAIN), 'openwakeword train.py missing after clone'
print(f'  ✓ openwakeword: {OWW_TRAIN} ({os.path.getsize(OWW_TRAIN)/1e3:.0f} KB)')

# ── Make sure openwakeword's PARENT dir is on sys.path so the package
# resolves correctly. /content is also on sys.path by default in Colab,
# which would resolve `import openwakeword` to /content/openwakeword/
# (the editable-install ROOT, no __init__.py) instead of the package
# at /content/openwakeword/openwakeword/. That would treat openwakeword
# as a namespace package and submodule imports (.data, .utils) would fail.
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)
# Wipe any stale namespace-package state in sys.modules
for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]
import openwakeword
assert openwakeword.__file__ is not None, (
    f'openwakeword still loaded as namespace package — '
    f'__file__ is None, __path__={openwakeword.__path__}. '
    f'Check sys.path for stray /content entries.'
)
print(f'  ✓ openwakeword importable: {openwakeword.__file__}')

# ── Download TTS voice models ──────────────────────────────────────────────────

import os, requests

PSG_DIR = '/content/piper-sample-generator'
VOICES_DIR = '/content/piper-voices'
os.makedirs(f'{PSG_DIR}/models', exist_ok=True)
os.makedirs(VOICES_DIR, exist_ok=True)

if LANGUAGE == 'en_US':
    # ── English: use libritts .pt model (904 speakers, best variety) ───
    PIPER_MODEL = f'{PSG_DIR}/models/en_US-libritts_r-medium.pt'
    PIPER_MODEL_URL = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
    if not os.path.exists(PIPER_MODEL) or os.path.getsize(PIPER_MODEL) < 100_000_000:
        print(f'  libritts model missing/partial — downloading (~200 MB)')
        !wget -q --tries=5 --timeout=300 -O {PIPER_MODEL} {PIPER_MODEL_URL}
    assert os.path.getsize(PIPER_MODEL) > 100_000_000, f'libritts model only {os.path.getsize(PIPER_MODEL)} bytes — download failed'
    print(f'  ✓ libritts model: {os.path.getsize(PIPER_MODEL)/1e6:.0f} MB ({LANGUAGE}, 904 speakers)')
    VOICE_MODELS = PIPER_MODEL
else:
    # ── Other languages: auto-discover Piper .onnx voices ──────────────
    VOICES_JSON_URL = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/voices.json'
    VOICES_BASE_URL = 'https://huggingface.co/rhasspy/piper-voices/resolve/v1.0.0/'
    QUALITY_RANK = {'high': 0, 'medium': 1, 'low': 2, 'x_low': 3}

    print(f'  Fetching voices.json for {LANGUAGE}...')
    voices = requests.get(VOICES_JSON_URL, timeout=30).json()
    matches = {k: v for k, v in voices.items() if v['language']['code'] == LANGUAGE}
    if not matches:
        avail = sorted({v['language']['code'] for v in voices.values()})
        raise ValueError(f'No Piper voices for "{LANGUAGE}". Available: {", ".join(avail)}')

    sorted_voices = sorted(matches.values(), key=lambda v: QUALITY_RANK.get(v['quality'], 9))
    onnx_files = []
    for voice in sorted_voices:
        for fpath, finfo in voice['files'].items():
            if not (fpath.endswith('.onnx') or fpath.endswith('.onnx.json')):
                continue
            local = f'{VOICES_DIR}/{os.path.basename(fpath)}'
            if os.path.exists(local) and os.path.getsize(local) == finfo['size_bytes']:
                if fpath.endswith('.onnx') and not fpath.endswith('.onnx.json'):
                    onnx_files.append(local)
                continue
            url = VOICES_BASE_URL + fpath
            !wget -q --tries=3 --timeout=120 -O "{local}" "{url}"
            if fpath.endswith('.onnx') and not fpath.endswith('.onnx.json'):
                onnx_files.append(local)
            tag = f'{voice["name"]}({voice["quality"]})'
            print(f'  ✓ {tag}: {os.path.basename(fpath)} ({os.path.getsize(local)/1e6:.1f} MB)')

    assert onnx_files, f'No .onnx voice files downloaded for {LANGUAGE}'
    lang_name = sorted_voices[0]['language']['name_english']
    print(f'  ✓ {len(onnx_files)} voice(s) for {LANGUAGE} ({lang_name})')
    VOICE_MODELS = onnx_files

# ── Apply runtime patches ─────────────────────────────────────────────────────

import os, glob

# Patch A: torch_audiomentations 0.11 calls torchaudio.set_audio_backend
# which torchaudio 2.x removed. sed-replace the bad line with no-op.
for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    !sed -i 's|torchaudio.set_audio_backend("soundfile")|pass  # patched|' "{path}"
    print(f'  Patch A (set_audio_backend): {path}')

# Patch B: write generate_samples.py shim that routes .pt models to
# piper-sample-generator's generate_samples() and .onnx models to
# generate_samples_onnx(). Uses VOICE_MODELS from cell 2b.
SHIM_SRC = '/content/piper-sample-generator/generate_samples.py'
SHIM_DST = '/content/openwakeword/openwakeword/generate_samples.py'
shim_code = f'''\
from piper_sample_generator.__main__ import generate_samples as _gen_pt
from piper_sample_generator.__main__ import generate_samples_onnx as _gen_onnx

_DEFAULT_MODEL = {repr(VOICE_MODELS)}

def generate_samples(*, model=_DEFAULT_MODEL, text, max_samples, output_dir,
                     file_names, noise_scales=None, noise_scale_ws=None,
                     length_scales=None, **kwargs):
    shared = dict(text=text, max_samples=max_samples, output_dir=output_dir,
                  file_names=file_names)
    if noise_scales: shared["noise_scales"] = noise_scales
    if noise_scale_ws: shared["noise_scale_ws"] = noise_scale_ws
    if length_scales: shared["length_scales"] = length_scales

    if isinstance(model, list) or (isinstance(model, str) and model.endswith(".onnx")):
        return _gen_onnx(model=model if isinstance(model, list) else [model], **shared)
    else:
        return _gen_pt(model=model, batch_size=kwargs.pop("batch_size", 50), **shared, **kwargs)
'''
with open(SHIM_SRC, 'w') as f:
    f.write(shim_code)
!cp "{SHIM_SRC}" "{SHIM_DST}"
print(f'  Patch B (generate_samples shim): {SHIM_DST}')
if isinstance(VOICE_MODELS, list):
    print(f'    routing to generate_samples_onnx ({len(VOICE_MODELS)} .onnx voices)')
else:
    print(f'    routing to generate_samples (.pt model)')

# Patch C: HF Hub default 10s timeouts -> 120s
os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
import huggingface_hub.constants as hfc
for attr in ['DEFAULT_ETAG_TIMEOUT', 'DEFAULT_DOWNLOAD_TIMEOUT',
             'HF_HUB_ETAG_TIMEOUT', 'HF_HUB_DOWNLOAD_TIMEOUT']:
    if hasattr(hfc, attr): setattr(hfc, attr, 120)
print('  Patch C (HF Hub timeouts -> 120s)')

# Patch D: torchaudio.info shim (removed in torchaudio 2.x)
import torchaudio
init_path = torchaudio.__file__
SHIM_MARKER = '# -- PATCH: info() shim for torchaudio 2.x --'
with open(init_path) as f:
    content = f.read()
if SHIM_MARKER not in content:
    shim = (f'\n\n{SHIM_MARKER}\n'
            'def info(file_path, *args, **kwargs):\n'
            '    import soundfile as _sf\n'
            '    si = _sf.info(str(file_path))\n'
            '    return type("_TorchaudioInfo", (), {\n'
            '        "num_frames": si.frames, "sample_rate": si.samplerate,\n'
            '        "num_channels": si.channels, "bits_per_sample": 16,\n'
            '        "encoding": "PCM_S",\n'
            '    })()\n')
    with open(init_path, 'a') as f: f.write(shim)
    import importlib; importlib.reload(torchaudio)
print(f'  Patch D (torchaudio.info shim): {hasattr(torchaudio, "info")}')

# Patch E: train.py val dtype cast (line ~519 + ~547)
TRAIN_PY = '/content/openwakeword/openwakeword/train.py'
!sed -i 's|val_predictions = self.model(x_val)$|val_predictions = self.model(x_val.float())|' "{TRAIN_PY}"
print('  Patch E (train.py val dtype cast)')
print()
print('All 5 patches applied (idempotent).')

_STEP_2_OK = True

## 3. Pre-flight checks

Hard-fails if anything's wrong. This is the gate before slow downloads start.

In [ ]:
assert '_STEP_2_OK' in dir(), '⚠️ Run cell 2 (Clone + voices + patches) first!'

import os, importlib

# Files
expected = {
    'piper-sample-generator/generate_samples.py': '/content/piper-sample-generator/generate_samples.py',
    'openwakeword/generate_samples.py':            '/content/openwakeword/openwakeword/generate_samples.py',
    'openwakeword/train.py':                       '/content/openwakeword/openwakeword/train.py',
    'openwakeword/data.py':                        '/content/openwakeword/openwakeword/data.py',
    'openwakeword/utils.py':                       '/content/openwakeword/openwakeword/utils.py',
}
# Add voice model check based on VOICE_MODELS from cell 2b
if isinstance(VOICE_MODELS, str):
    expected['voice model (.pt)'] = VOICE_MODELS
elif isinstance(VOICE_MODELS, list):
    for i, vm in enumerate(VOICE_MODELS):
        expected[f'voice model {i+1} (.onnx)'] = vm

missing_files = []
for name, p in expected.items():
    if os.path.exists(p):
        size = os.path.getsize(p)
        unit = 'MB' if size > 1e6 else 'KB'
        denom = 1e6 if unit == 'MB' else 1e3
        print(f'  ✓ {name}: {size/denom:.1f} {unit}')
    else:
        missing_files.append((name, p))
        print(f'  ✗ MISSING: {name} ({p})')

# Imports
missing_mods = []
for mod in ['torch', 'torchinfo', 'torchmetrics', 'scipy', 'numpy', 'tqdm',
            'yaml', 'mutagen', 'pronouncing', 'torchaudio', 'audiomentations',
            'torch_audiomentations', 'speechbrain', 'acoustics', 'onnx',
            'onnxruntime', 'soundfile', 'requests', 'huggingface_hub',
            'piper_phonemize', 'piper', 'piper_sample_generator', 'openwakeword']:
    try:
        importlib.import_module(mod)
    except Exception as e:
        missing_mods.append((mod, str(e)))
        print(f'  ✗ IMPORT FAIL: {mod} ({type(e).__name__}: {e})')

# Dry-run openwakeword.data + utils — catches weird internal import bugs
try:
    from openwakeword.data import generate_adversarial_texts, augment_clips, mmap_batch_generator
    from openwakeword.utils import compute_features_from_generator, AudioFeatures
    print('  ✓ openwakeword.data + .utils dry-import OK')
except Exception as e:
    print(f'  ✗ openwakeword internal import: {type(e).__name__}: {e}')
    missing_mods.append(('openwakeword.data/utils', str(e)))

import torch
print(f'\n  CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}')
if missing_files or missing_mods:
    raise RuntimeError(f'Pre-flight failed — {len(missing_files)} missing files, {len(missing_mods)} import errors. Re-run cells 1-3.')
print('\n  Pre-flight PASSED. Safe to proceed with downloads + training.')

_STEP_3_OK = True

## 4. Download shared models + training data (~15 min)

Downloads openwakeword embedding models, MIT impulse responses, FMA music, and ACAV features. All with resume support.

In [ ]:
assert '_STEP_3_OK' in dir(), '⚠️ Run cell 3 (Pre-flight) first!'

# ── openwakeword shared models (mel + embedding) ────────────────────────

import os
models_dir = '/content/openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    out = f'{models_dir}/{fname}'
    if os.path.exists(out) and os.path.getsize(out) > 1000:
        print(f'  cached: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')
    else:
        !wget -q --tries=5 --timeout=120 "{BASE}/{fname}" -O "{out}"
        print(f'  âœ“ downloaded: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')

# ── MIT impulse responses ─────────────────────────────────────────────────

import os, time, numpy as np
import scipy.io.wavfile as wavfile
from huggingface_hub import snapshot_download
import datasets

output_dir = '/content/mit_rirs'
os.makedirs(output_dir, exist_ok=True)
if len([f for f in os.listdir(output_dir) if f.endswith('.wav')]) >= 250:
    print(f'  cached: {len(os.listdir(output_dir))} MIT IR WAVs')
else:
    # Pre-cache via snapshot_download (more reliable than load_dataset's internal
    # 10s timeout on 270 file metadata calls).
    for i in range(6):
        try:
            snapshot_download(repo_id='davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', etag_timeout=120, max_workers=4)
            break
        except Exception as e:
            wait = 15 * (i+1)
            print(f'  snapshot_download attempt {i+1}/6: {type(e).__name__}: {str(e)[:80]} â€” sleep {wait}s')
            time.sleep(wait)
    # Load + convert
    for attempt in range(4):
        try:
            rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                                split='train', streaming=False)
            break
        except Exception as e:
            print(f'  load_dataset attempt {attempt+1}: {type(e).__name__}: {str(e)[:120]}')
            time.sleep(20 * (attempt + 1))
    else:
        raise RuntimeError('load_dataset failed for MIT IRs')
    n = 0
    for i, row in enumerate(rir_dataset):
        out = f'{output_dir}/{i:04d}.wav'
        if os.path.exists(out): continue
        audio = row['audio']
        sr = audio['sampling_rate']
        arr = np.asarray(audio['array'])
        if sr != 16000:
            from scipy.signal import resample_poly
            arr = resample_poly(arr, 16000, sr)
        arr = (arr * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(out, 16000, arr); n += 1
    print(f'  âœ“ wrote {n} new WAVs (total {len(os.listdir(output_dir))})')

# ── FMA + ACAV features ───────────────────────────────────────────────────

import os, requests, zipfile
from tqdm.auto import tqdm
os.chdir('/content')

# â”€â”€ ACAV features (17 GB) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ACAV_FULL = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if os.path.exists(ACAV_FULL) and os.path.getsize(ACAV_FULL) > 16_000_000_000:
    print(f'  cached: ACAV {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')
else:
    resume_byte = os.path.getsize(ACAV_FULL) if os.path.exists(ACAV_FULL) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  ACAV: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(ACAV_URL, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(ACAV_FULL, mode) as f, tqdm(total=total, initial=resume_byte,
                                              unit='B', unit_scale=True, unit_divisor=1024,
                                              desc='ACAV') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  âœ“ ACAV: {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')

# â”€â”€ FMA small (~8 GB zip) with resume + extract â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FMA_DIR = '/content/fma'
FMA_ZIP = '/content/fma_small.zip'
os.makedirs(FMA_DIR, exist_ok=True)
if os.path.isdir(FMA_DIR) and len(os.listdir(FMA_DIR)) > 100:
    print(f'  cached: FMA {len(os.listdir(FMA_DIR))} entries')
else:
    fma_url = 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'
    resume_byte = os.path.getsize(FMA_ZIP) if os.path.exists(FMA_ZIP) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  FMA: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(fma_url, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(FMA_ZIP, mode) as f, tqdm(total=total, initial=resume_byte,
                                            unit='B', unit_scale=True, unit_divisor=1024,
                                            desc='FMA') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  extracting FMA...')
    with zipfile.ZipFile(FMA_ZIP) as zf:
        for name in tqdm(zf.namelist(), desc='extract', unit='f'):
            zf.extract(name, FMA_DIR)
    os.remove(FMA_ZIP)
    print(f'  âœ“ FMA extracted: {len(os.listdir(FMA_DIR))} entries')

_STEP_4_OK = True

## 5. Prepare audio data (~6 min)

Converts FMA MP3s to 16-kHz mono WAVs and subsamples ACAV features.

In [ ]:
assert '_STEP_4_OK' in dir(), '⚠️ Run cell 4 (Downloads) first!'

# ── FMA MP3 → 16-kHz mono WAV ─────────────────────────────────────────────

import os, glob, subprocess
from tqdm.auto import tqdm
out_dir = '/content/fma_wav'
os.makedirs(out_dir, exist_ok=True)
n_existing = len(os.listdir(out_dir))
if n_existing >= 1500:
    print(f'  cached: {n_existing} FMA WAVs')
else:
    mp3s = glob.glob('/content/fma/**/*.mp3', recursive=True)
    if len(mp3s) < 100:
        raise RuntimeError(f'FMA download incomplete â€” only {len(mp3s)} MP3s found')
    existing = set(os.listdir(out_dir))
    n_target = min(1500, len(mp3s))
    to_convert = [(i, mp3s[i]) for i in range(n_target) if f'{i:05d}.wav' not in existing]
    print(f'  Converting {len(to_convert)} MP3s â†’ 16kHz mono WAVs (skipping {n_target - len(to_convert)} cached)')
    for i, mp3 in tqdm(to_convert):
        out = f'{out_dir}/{i:05d}.wav'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', mp3,
                        '-ac', '1', '-ar', '16000', out], check=False)
    print(f'  âœ“ {len(os.listdir(out_dir))} FMA WAVs ready')

# ── Subsample ACAV ────────────────────────────────────────────────────────

import numpy as np, os
SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
TRAIN_DST = '/content/acav_train_subset.npy'
VAL_DST   = '/content/acav_val_subset.npy'
if os.path.exists(TRAIN_DST) and os.path.exists(VAL_DST):
    train_arr = np.load(TRAIN_DST, mmap_mode='r')
    val_arr = np.load(VAL_DST, mmap_mode='r')
    print(f'  cached: train={train_arr.shape} ({train_arr.nbytes/1e9:.2f} GB), val={val_arr.shape} ({val_arr.nbytes/1e6:.0f} MB)')
else:
    if not os.path.exists(SRC):
        raise RuntimeError(f'ACAV source missing: {SRC} â€” re-run cell 7')
    arr = np.load(SRC, mmap_mode='r')
    print(f'  source: {arr.shape}, dtype={arr.dtype}')
    n_total = len(arr)
    n_train = n_total // 10
    n_val_rows = n_total // 100
    print(f'  slicing: train=[:{n_train}] (3-D), val=[{n_train}:{n_train+n_val_rows}] (flattened to 2-D)')
    train_chunk = np.array(arr[:n_train])
    np.save(TRAIN_DST, train_chunk)
    print(f'  âœ“ train: {train_chunk.shape} ({train_chunk.nbytes/1e9:.2f} GB)')
    del train_chunk
    val_chunk = np.array(arr[n_train:n_train+n_val_rows])
    val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
    np.save(VAL_DST, val_flat)
    print(f'  âœ“ val: {val_flat.shape} ({val_flat.nbytes/1e6:.0f} MB)')
    del val_chunk, val_flat
    os.remove(SRC)
    print('  removed 17 GB original')

_STEP_5_OK = True

## 6. Build config + generate TTS clips (~15 min)

Builds training YAML config, patches deep-phonemizer, generates positive/negative clips via Piper TTS, and resamples to 16 kHz.

In [ ]:
assert '_STEP_5_OK' in dir(), '⚠️ Run cell 5 (Prepare audio) first!'

# ── Build training config ─────────────────────────────────────────────────

import yaml, os

# TARGET_PHRASE, MODEL_NAME, and LANGUAGE are defined in cell 0 (user config).
# This cell builds the training config dict from those variables.

config = {
    # ── Wake-word ───────────────────────────────────────────────────────
    'target_phrase':   TARGET_PHRASE,
    'model_name':      MODEL_NAME,
    'custom_negative_phrases': [],

    # ── Synthetic clip generation ───────────────────────────────────────
    'n_samples':       2000,
    'n_samples_val':   1000,
    'tts_batch_size':  50,
    'piper_sample_generator_path': '/content/piper-sample-generator',

    # ── Augmentation ────────────────────────────────────────────────────
    'augmentation_rounds':    1,
    'augmentation_batch_size': 16,

    # ── Training (read by hand-rolled trainer in cell 14) ──────────────
    'steps':           20000,
    'max_negative_weight': 1500,
    'target_accuracy': 0.7,
    'target_recall':   0.5,
    'target_false_positives_per_hour': 0.5,
    'batch_size':      128,
    'learning_rate':   1e-4,

    # ── Model arch (matches openwakeword's DNN exactly) ────────────────
    'model_type':      'dnn',
    'layer_dim':       128,
    'layer_size':      128,
    'n_blocks':        1,
    'model_input_shape': [16, 96],
    'n_classes':       1,
    'batch_n_per_class': {
        'ACAV100M_sample':       1024,
        'adversarial_negative':    50,
        'positive':                50,
    },

    # ── Data paths ──────────────────────────────────────────────────────
    'background_paths': ['/content/fma_wav'],
    'background_paths_duplication_rate': [1],
    'rir_paths':        ['/content/mit_rirs'],
    'false_positive_validation_data_path': '/content/acav_val_subset.npy',
    'feature_data_files': {'ACAV100M_sample': '/content/acav_train_subset.npy'},
    'output_dir':       f'/content/{MODEL_NAME}_output',
    'tflite_export':    True,
    'onnx_export':      True,

    # ── Clip dirs (used by hand-rolled trainer) ────────────────────────
    'positive_clips_train_dir': f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_train',
    'positive_clips_test_dir':  f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_test',
    'negative_clips_train_dir': f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_train',
    'negative_clips_test_dir':  f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_test',
    'feature_save_dir':         f'/content/{MODEL_NAME}_output/{MODEL_NAME}',
}
os.makedirs(config['output_dir'], exist_ok=True)
with open('/content/my_model.yaml', 'w') as f:
    yaml.dump(config, f, sort_keys=False)
print(f'  target_phrase: {config["target_phrase"]}')
print(f'  model_name:    {config["model_name"]}')
print(f'  language:      {LANGUAGE}')
print(f'  steps:         {config["steps"]}')
print(f'  max_neg_w:     {config["max_negative_weight"]}')
print(f'  target_recall: {config["target_recall"]}')
print(f'  target_fp/hr:  {config["target_false_positives_per_hour"]}')

# ── Patch deep-phonemizer ─────────────────────────────────────────────────

# Workaround: deep-phonemizer's torch.load() doesn't pass weights_only=False,
# which PyTorch 2.x now requires for loading pickled checkpoints. Upstream fix
# would be deep-phonemizer adding weights_only=False to its own torch.load call.
# Until then, we sed-patch the installed file.
!pip install -q deep-phonemizer
import glob
for path in glob.glob('/usr/local/lib/python*/dist-packages/dp/model/model.py'):
    !sed -i 's/torch.load(checkpoint_path, map_location=device)/torch.load(checkpoint_path, map_location=device, weights_only=False)/g' "{path}"
    print(f'  Patched: {path}')
print('  deep-phonemizer torch.load patched for PyTorch 2.x')

# ── Generate Piper TTS clips ──────────────────────────────────────────────

import sys, os, yaml
# Idempotent: check ALL FOUR clip dirs and skip the generate_clips invocation
# only if every dir is fully populated. Upstream's `n_current_samples <= 0.95
# *n_samples` checks at lines 675/692/706/729 in train.py also skip cached
# dirs internally, so re-running is safe even if some dirs are partial.
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
dirs = {
    'positive_train': (cfg['positive_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'positive_test':  (cfg['positive_clips_test_dir'],  int(cfg['n_samples_val'] * 0.75)),
    'negative_train': (cfg['negative_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'negative_test':  (cfg['negative_clips_test_dir'],  int(cfg['n_samples_val'] * 0.75)),
}
status = []
for name, (path, expected) in dirs.items():
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    status.append((name, n, expected, n >= expected))
all_full = all(ok for _, _, _, ok in status)
for name, n, expected, ok in status:
    print(f'  {"PASS" if ok else "MISS"} {name}: {n} clips (expect >={expected})')
if all_full:
    print('\n  cached: all 4 clip dirs already populated')
else:
    print('\n  -> running generate_clips (upstream skips cached dirs internally)')
    !{sys.executable} /content/openwakeword/openwakeword/train.py \
        --training_config /content/my_model.yaml \
        --generate_clips
    for name, (path, expected) in dirs.items():
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        assert n >= expected, f'generate_clips left {name} with only {n} clips (expected >={expected})'
    print('\n  PASS all 4 clip dirs now populated')

# ── Resample TTS clips to 16 kHz ─────────────────────────────────────────

import os, glob, soundfile as sf, yaml
from scipy.signal import resample_poly
from tqdm.auto import tqdm
TARGET_SR = 16000
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
ROOT = cfg['output_dir']
wav_dirs = sorted({os.path.dirname(f)
                   for f in glob.glob(f'{ROOT}/**/*.wav', recursive=True)})
for d in wav_dirs:
    files = [f for f in os.listdir(d) if f.endswith('.wav')]
    if not files: continue
    sr_counts = {}
    for f in files[:20]:
        sr = sf.info(f'{d}/{f}').samplerate
        sr_counts[sr] = sr_counts.get(sr, 0) + 1
    if all(k == TARGET_SR for k in sr_counts):
        print(f'  ok {d}: {len(files)} files at 16 kHz (skip)')
        continue
    print(f'  resampling {d}: {len(files)} files, SRs {sr_counts}')
    n = 0
    for f in tqdm(files, desc=os.path.basename(d) or d, leave=False):
        p = f'{d}/{f}'
        data, sr = sf.read(p)
        if sr == TARGET_SR: continue
        new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
        sf.write(p, new_data, TARGET_SR); n += 1
    print(f'  ok {d}: resampled {n}/{len(files)}')
# Clear stale .npy features so augment re-runs cleanly
for f in glob.glob(f'{ROOT}/**/*.npy', recursive=True):
    os.remove(f)
print('\n  cleared stale features (will be regenerated by augment)')

_STEP_6_OK = True

## 7. Augment + train (~40 min)

Augments clips with background noise + reverb, extracts features, then runs 3-stage training with FP/hour validation.

In [ ]:
assert '_STEP_6_OK' in dir(), '⚠️ Run cell 6 (Config + clips) first!'

# ── Augment + featurise ───────────────────────────────────────────────────

import sys, os, glob, yaml
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
FEAT = cfg['feature_save_dir']
needed = ['positive_features_train.npy', 'negative_features_train.npy',
          'positive_features_test.npy',  'negative_features_test.npy']
if all(os.path.exists(f'{FEAT}/{n}') for n in needed):
    print('  cached: all 4 feature .npy files exist')
else:
    !{sys.executable} /content/openwakeword/openwakeword/train.py \
        --training_config /content/my_model.yaml \
        --augment_clips
for f in sorted(glob.glob(f'{FEAT}/*.npy')):
    print(f'  {f}: {os.path.getsize(f)/1e6:.1f} MB')
for n in needed:
    assert os.path.exists(f'{FEAT}/{n}'), f"augment didn't produce {n}"
print('  PASS all 4 feature files present')

# ── Hand-rolled trainer ────────────────────────────────────────────────────

import os, copy, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
FEAT = cfg['feature_save_dir']
pos_train = torch.from_numpy(np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)).to(DEVICE)
neg_train = torch.from_numpy(np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)).to(DEVICE)
pos_test  = torch.from_numpy(np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)).to(DEVICE)
neg_test  = torch.from_numpy(np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)).to(DEVICE)
print(f'  pos_train={tuple(pos_train.shape)}  neg_train={tuple(neg_train.shape)}')
print(f'  pos_test={tuple(pos_test.shape)}  neg_test={tuple(neg_test.shape)}')

acav_train_np = np.load(cfg['feature_data_files']['ACAV100M_sample'], mmap_mode='r')
print(f'  acav_train (mmap): {acav_train_np.shape}')
acav_val_np = np.load(cfg['false_positive_validation_data_path'])
M = acav_val_np.shape[0]
val_listen_hours = M * 0.08 / 3600.0
print(f'  acav_val: {acav_val_np.shape}  â†’ {val_listen_hours:.2f} hr listening time')
n_win_val = M - 16
acav_val_windows = np.lib.stride_tricks.sliding_window_view(acav_val_np, (16, 96))[:, 0, :, :]
acav_val_windows = np.ascontiguousarray(acav_val_windows.astype(np.float32))
print(f'  acav_val_windows: {acav_val_windows.shape} ({acav_val_windows.nbytes/1e6:.0f} MB)')
VAL_BATCH = 4096

class WakewordModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layer1 = nn.Linear(16 * 96, 128)
        self.layernorm1 = nn.LayerNorm(128)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(128, 1)
    def forward(self, x):
        return self.layer2(self.relu1(self.layernorm1(self.layer1(self.flatten(x)))))

model = WakewordModel().to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss(reduction='none')
TOTAL_STEPS = cfg['steps']
MAX_NEG_W   = cfg['max_negative_weight']
TARGET_FP_PER_HR = cfg['target_false_positives_per_hour']
THRESH = 0.5
history = {'val_recall': [], 'val_accuracy': [], 'val_fp_per_hour': [], 'val_n_fp': [], 'loss': []}
best_models = []

B_POS, B_ANEG, B_ACAV = 32, 32, 64
def random_acav_window_batch(k):
    N, T, F = acav_train_np.shape
    rows = np.random.randint(0, N, size=k)
    starts = np.random.randint(0, T - 16 + 1, size=k)
    out = np.empty((k, 16, F), dtype=np.float32)
    for i, (r, s) in enumerate(zip(rows, starts)):
        out[i] = acav_train_np[r, s:s+16, :].astype(np.float32)
    return out

def build_batch():
    p_idx = torch.randint(0, pos_train.shape[0], (B_POS,), device=DEVICE)
    aneg_idx = torch.randint(0, neg_train.shape[0], (B_ANEG,), device=DEVICE)
    p, an = pos_train[p_idx], neg_train[aneg_idx]
    acav = torch.from_numpy(random_acav_window_batch(B_ACAV)).to(DEVICE)
    x = torch.cat([p, an, acav], dim=0)
    y = torch.cat([torch.ones(B_POS, device=DEVICE),
                   torch.zeros(B_ANEG + B_ACAV, device=DEVICE)])
    return x, y

@torch.no_grad()
def validate(step_label):
    model.eval()
    p_preds = torch.sigmoid(model(pos_test)).squeeze(-1)
    n_preds = torch.sigmoid(model(neg_test)).squeeze(-1)
    recall = (p_preds >= THRESH).float().mean().item()
    accuracy = (((p_preds >= THRESH).sum() + (n_preds < THRESH).sum()).item()
                / (pos_test.shape[0] + neg_test.shape[0]))
    n_fp = 0
    for i in range(0, n_win_val, VAL_BATCH):
        chunk = torch.from_numpy(acav_val_windows[i:i+VAL_BATCH]).to(DEVICE)
        n_fp += (torch.sigmoid(model(chunk)).squeeze(-1) >= THRESH).sum().item()
    fp_per_hour = n_fp / max(val_listen_hours, 1e-6)
    history['val_recall'].append(recall)
    history['val_accuracy'].append(accuracy)
    history['val_fp_per_hour'].append(fp_per_hour)
    history['val_n_fp'].append(n_fp)
    save = False
    if len(history['val_n_fp']) >= 3:
        fp_p50 = np.percentile(history['val_n_fp'], 50)
        rc_p5  = np.percentile(history['val_recall'], 5)
        if n_fp <= fp_p50 and recall >= rc_p5:
            best_models.append((copy.deepcopy(model.state_dict()),
                                {'val_recall': recall, 'val_accuracy': accuracy,
                                 'val_fp_per_hour': fp_per_hour, 'val_n_fp': n_fp}))
            save = True
    print(f'  [{step_label}] recall={recall:.3f}  acc={accuracy:.3f}  fp/hr={fp_per_hour:.2f}  saved={"+" if save else "-"}', flush=True)
    model.train()

def run_stage(stage_idx, n_steps, lr, max_neg_w, val_window_frac=1.0):
    optimizer = optim.Adam(model.parameters(), lr=lr)
    weight_schedule = np.linspace(1.0, max_neg_w, n_steps)
    val_start = int(n_steps * (1.0 - val_window_frac))
    val_steps = set(np.linspace(val_start, n_steps - 1, 20).astype(int))
    warmup = max(1, n_steps // 5)
    hold   = n_steps // 3
    accumulated = []
    print(f'\n=== Stage {stage_idx}: {n_steps} steps, lr={lr}, max_neg_w={max_neg_w} ===', flush=True)
    t0 = time.time()
    for step in range(n_steps):
        if step < warmup:
            lr_now = lr * (step + 1) / warmup
        elif step < warmup + hold:
            lr_now = lr
        else:
            decay_t = (step - warmup - hold) / max(1, n_steps - warmup - hold)
            lr_now = lr * 0.5 * (1.0 + math.cos(math.pi * min(1.0, decay_t)))
        for pg in optimizer.param_groups: pg['lr'] = lr_now
        x, y = build_batch()
        logits = model(x).squeeze(-1)
        preds = torch.sigmoid(logits)
        keep = ((y == 0) & (preds >= 0.001)) | ((y == 1) & (preds < 0.999))
        if keep.sum() == 0:
            if step in val_steps: validate(f'stage{stage_idx} step {step}/{n_steps}')
            continue
        kept_logits = logits[keep]
        kept_y = y[keep]
        neg_w = weight_schedule[step]
        w = torch.where(kept_y > 0.5,
                        torch.tensor(1.0, device=DEVICE),
                        torch.tensor(neg_w, device=DEVICE, dtype=torch.float32))
        accumulated.append((kept_logits, kept_y, w))
        if sum(t[0].shape[0] for t in accumulated) >= 128:
            cat_logits = torch.cat([t[0] for t in accumulated])
            cat_y = torch.cat([t[1] for t in accumulated])
            cat_w = torch.cat([t[2] for t in accumulated])
            loss = (loss_fn(cat_logits, cat_y) * cat_w).mean()
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            history['loss'].append(loss.item())
            accumulated.clear()
        if step in val_steps:
            elapsed = (time.time() - t0) / 60
            validate(f'stage{stage_idx} step {step}/{n_steps} ({elapsed:.1f}m, lr={lr_now:.6f}, neg_w={neg_w:.0f})')
    print(f'  Stage {stage_idx} done in {(time.time()-t0)/60:.1f} min', flush=True)

stage1_steps = TOTAL_STEPS
stage2_steps = max(2000, TOTAL_STEPS // 10)
stage3_steps = max(2000, TOTAL_STEPS // 10)
max_neg_w_now = MAX_NEG_W
run_stage(1, stage1_steps, lr=1e-4, max_neg_w=max_neg_w_now, val_window_frac=0.25)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage1 best FP/hr > target â†’ max_neg_w doubled to {max_neg_w_now}')
run_stage(2, stage2_steps, lr=1e-5, max_neg_w=max_neg_w_now, val_window_frac=1.0)
if (history['val_fp_per_hour'] and min(history['val_fp_per_hour']) > TARGET_FP_PER_HR):
    max_neg_w_now *= 2
    print(f'\n[adapt] stage2 best FP/hr > target â†’ max_neg_w doubled to {max_neg_w_now}')
run_stage(3, stage3_steps, lr=1e-6, max_neg_w=max_neg_w_now, val_window_frac=1.0)
print(f'\n=== Training done. {len(best_models)} checkpoints saved. ===')
print(f'  best val_fp_per_hour: {min(history["val_fp_per_hour"]):.2f}')
print(f'  best val_recall:      {max(history["val_recall"]):.3f}')
print(f'  best val_accuracy:    {max(history["val_accuracy"]):.3f}')

_STEP_7_OK = True

## 8. Export + download

Ensemble best checkpoints, export ONNX (sigmoid baked in) + TFLite (via Keras weight transfer), verify numerical equivalence, browser-download.

In [ ]:
assert '_STEP_7_OK' in dir(), '⚠️ Run cell 7 (Augment + train) first!'

import copy, os
import numpy as np
import torch
if not best_models:
    print('  no saved checkpoints — using current model state')
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s['val_accuracy'] for _, s in best_models]
    rcs  = [s['val_recall']   for _, s in best_models]
    fps  = [s['val_fp_per_hour'] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90  = np.percentile(rcs, 90)
    fp_p10  = np.percentile(fps, 10)
    print(f'  thresholds: acc≥{acc_p90:.3f}  recall≥{rc_p90:.3f}  fp/hr≤{fp_p10:.2f}')
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                 and sc['val_fp_per_hour'] <= fp_p10]
    print(f'  qualified: {len(qualified)} of {len(best_models)} checkpoints')
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))
        qualified = [sorted_models[0]]
        print('  no checkpoint qualified — fell back to single best')
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    qa = np.mean([s['val_accuracy'] for _, s in qualified])
    qr = np.mean([s['val_recall'] for _, s in qualified])
    qf = np.mean([s['val_fp_per_hour'] for _, s in qualified])
    print(f'  ensemble averaged: mean acc={qa:.3f}, recall={qr:.3f}, fp/hr={qf:.2f}')
model.load_state_dict(final_state)
model.eval()

# ── ONNX export (sigmoid baked in so APK runtime sees [0, 1]) ──────
out_path = f"{cfg['feature_save_dir']}/{cfg['model_name']}.onnx"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
class WakewordExportable(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x): return torch.sigmoid(self.base(x))
export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)
torch.onnx.export(export_model, dummy, out_path,
                  input_names=['onnx::Flatten_0'], output_names=['output'],
                  dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                  opset_version=14, dynamo=False)
print(f'  ✓ wrote {out_path} ({os.path.getsize(out_path)/1e3:.0f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(out_path)
print(f'  input:  {sess.get_inputs()[0].name} {sess.get_inputs()[0].shape}')
print(f'  output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}')
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    onnx_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f'  positive test set: mean={onnx_scores.mean():.3f}, recall@0.5={(onnx_scores>=0.5).mean():.3f}')
print(f'  ACAV val FP/hour at 0.5: {history["val_fp_per_hour"][-1]:.2f}')

# ── TFLite export (rebuild in Keras, transfer weights) ─────────────
tflite_path = None
if cfg.get('tflite_export', False):
    import tensorflow as tf
    with torch.no_grad():
        w1 = model.layer1.weight.cpu().numpy()
        b1 = model.layer1.bias.cpu().numpy()
        ln_gamma = model.layernorm1.weight.cpu().numpy()
        ln_beta  = model.layernorm1.bias.cpu().numpy()
        w2 = model.layer2.weight.cpu().numpy()
        b2 = model.layer2.bias.cpu().numpy()

    inputs = tf.keras.Input(shape=(16, 96), name='input')
    x = tf.keras.layers.Flatten()(inputs)
    x = tf.keras.layers.Dense(128, use_bias=True)(x)
    x = tf.keras.layers.LayerNormalization(epsilon=1e-5)(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Dense(1, activation='sigmoid', use_bias=True)(x)
    keras_model = tf.keras.Model(inputs, x)

    # Keras Dense: [in, out]; PyTorch Linear: [out, in] — transpose needed
    dense1 = keras_model.layers[2]
    dense1.set_weights([w1.T, b1])
    ln = keras_model.layers[3]
    ln.set_weights([ln_gamma, ln_beta])
    dense2 = keras_model.layers[5]
    dense2.set_weights([w2.T, b2])

    # Numerical equivalence check vs ONNX
    keras_scores = keras_model.predict(p_test_np, verbose=0).flatten()
    max_diff = np.max(np.abs(onnx_scores - keras_scores))
    print(f'  TFLite: Keras vs ONNX max abs diff = {max_diff:.2e}')
    assert max_diff < 1e-4, f'Keras/ONNX mismatch too large: {max_diff}'

    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    tflite_model = converter.convert()
    tflite_path = out_path.replace('.onnx', '.tflite')
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    print(f'  ✓ wrote {tflite_path} ({os.path.getsize(tflite_path)/1e3:.0f} KB)')

    # Verify TFLite interpreter output
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp_detail = interpreter.get_input_details()[0]
    out_detail = interpreter.get_output_details()[0]
    tflite_scores = np.empty(len(p_test_np), dtype=np.float32)
    for i in range(len(p_test_np)):
        interpreter.set_tensor(inp_detail['index'], p_test_np[i:i+1])
        interpreter.invoke()
        tflite_scores[i] = interpreter.get_tensor(out_detail['index']).flatten()[0]
    tflite_diff = np.max(np.abs(onnx_scores - tflite_scores))
    print(f'  TFLite: interpreter vs ONNX max abs diff = {tflite_diff:.2e}')
    print(f'  TFLite: positive test mean={tflite_scores.mean():.3f}, recall@0.5={(tflite_scores>=0.5).mean():.3f}')

# ── Download ───────────────────────────────────────────────────────
from google.colab import files
files.download(out_path)
if tflite_path:
    files.download(tflite_path)
print(f'\nDONE. Wake word "{cfg["model_name"]}" trained.')
print('Drop the downloaded model(s) into your app:')
print(f'  .onnx:   {cfg["model_name"]}.onnx')
if tflite_path:
    print(f'  .tflite: {cfg["model_name"]}.tflite')